In [0]:
%pip install databricks-labs-dqx
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.1/775.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.1/606.1 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2a20138c-5a32-49a2-8ae3-8ad312a0f175
    Can't uninstall 'PyYAML'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2a20138c-5a32-49a2-8ae3-8ad312a0f175
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: data

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.labs.dqx.profiler.generator import DQGenerator
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

ws = WorkspaceClient()
profiler = DQProfiler(ws)

# Load Chicago Bronze
chicago_bronze = spark.table("food_inspections.bronze.chicago_inspections")
print(f"Chicago Bronze: {chicago_bronze.count()} rows")

# Profile Chicago
print("\nProfiling Chicago Bronze...")
chi_summary_stats, chi_profiles = profiler.profile(chicago_bronze)

# Generate rule candidates
generator = DQGenerator(ws)
chi_checks = generator.generate_dq_rules(chi_profiles)

print(f"\nDQX generated {len(chi_checks)} rule candidates for Chicago:\n")
for rule in chi_checks:
    print(f"  {rule}")

Chicago Bronze: 308357 rows

Profiling Chicago Bronze...


00:14:06  INFO [d.l.d.profiler.generator] ✅ Quality rules generation completed. Generated 26 rules.



DQX generated 26 rule candidates for Chicago:

  {'check': {'function': 'is_not_null', 'arguments': {'column': 'inspection_id'}}, 'name': 'inspection_id_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'dba_name'}}, 'name': 'dba_name_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'aka_name'}}, 'name': 'aka_name_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'license_num'}}, 'name': 'license_num_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null_and_not_empty', 'arguments': {'column': 'facility_type', 'trim_strings': True}}, 'name': 'facility_type_is_null_or_empty', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'risk'}}, 'name': 'risk_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_in_list', 'arguments': {'column': 'risk', 'allowed': ['Risk 1 (High)', 'Risk 2 (

In [0]:
# Load Dallas Bronze
dallas_bronze = spark.table("food_inspections.bronze.dallas_inspections")
print(f"Dallas Bronze: {dallas_bronze.count()} rows")

# Profile Dallas (only core columns, not all 117)
# DQX profiling on 117 columns will take forever, subset to core cols
dallas_core = dallas_bronze.select(
    "restaurant_name", "inspection_type", "inspection_date",
    "inspection_score", "street_address", "zip_code",
    "lat_long_location", "source_city", "load_ts", "source_file",
    "violation_description_1", "violation_points_1",
)

print("\nProfiling Dallas Bronze (core columns)...")
dal_summary_stats, dal_profiles = profiler.profile(dallas_core)

dal_checks = generator.generate_dq_rules(dal_profiles)

print(f"\nDQX generated {len(dal_checks)} rule candidates for Dallas:\n")
for rule in dal_checks:
    print(f"  {rule}")

Dallas Bronze: 78984 rows

Profiling Dallas Bronze (core columns)...


00:14:59  INFO [d.l.d.profiler.generator] ✅ Quality rules generation completed. Generated 16 rules.



DQX generated 16 rule candidates for Dallas:

  {'check': {'function': 'is_not_null', 'arguments': {'column': 'restaurant_name'}}, 'name': 'restaurant_name_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'inspection_type'}}, 'name': 'inspection_type_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_in_list', 'arguments': {'column': 'inspection_type', 'allowed': ['Routine', 'Follow-up', 'Complaint']}}, 'name': 'inspection_type_other_value', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'inspection_date'}}, 'name': 'inspection_date_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'inspection_score'}}, 'name': 'inspection_score_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_null', 'arguments': {'column': 'street_address'}}, 'name': 'street_address_is_null', 'criticality': 'error'}
  {'check': {'function': 'is_not_nul

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule
from databricks.labs.dqx.check_funcs import (
    is_not_null_and_not_empty,
    regex_match,
    sql_expression,
)
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

ws = WorkspaceClient()
dq_engine = DQEngine(ws)

chicago_bronze = spark.table("food_inspections.bronze.chicago_inspections")

chicago_checks = [
    DQRowRule(
        name="restaurant_name_not_null",
        criticality="error",
        check_func=is_not_null_and_not_empty,
        column="dba_name",
    ),
    DQRowRule(
        name="inspection_date_not_null",
        criticality="error",
        check_func=is_not_null_and_not_empty,
        column="inspection_date",
    ),
    DQRowRule(
        name="inspection_type_not_null",
        criticality="error",
        check_func=is_not_null_and_not_empty,
        column="inspection_type",
    ),
    DQRowRule(
        name="zip_valid_format",
        criticality="error",
        check_func=regex_match,
        column="zip",
        check_func_kwargs={"regex": "^[0-9]{5}$"},
    ),
    DQRowRule(
        name="results_not_null",
        criticality="error",
        check_func=is_not_null_and_not_empty,
        column="results",
    ),
    DQRowRule(
        name="has_violations",
        criticality="error",
        check_func=is_not_null_and_not_empty,
        column="violations",
    ),
]

chicago_valid, chicago_invalid = dq_engine.apply_checks_and_split(chicago_bronze, chicago_checks)

valid_count = chicago_valid.count()
invalid_count = chicago_invalid.count()
print(f"Chicago valid:   {valid_count:,}")
print(f"Chicago invalid: {invalid_count:,}")
print(f"Total:           {valid_count + invalid_count:,}")

chicago_invalid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "food_inspections.silver.chicago_quarantine"
)
print(f"\nQuarantine saved: food_inspections.silver.chicago_quarantine ({invalid_count:,} rows)")

print("\nSample quarantined rows:")
chicago_invalid.select("inspection_id", "dba_name", "zip", "results", "violations", "_errors").show(5, truncate=60)

Chicago valid:   221,885
Chicago invalid: 86,472
Total:           308,357

Quarantine saved: food_inspections.silver.chicago_quarantine (86,472 rows)

Sample quarantined rows:
+-------------+-----------------------+-----+---------------+----------+------------------------------------------------------------+
|inspection_id|               dba_name|  zip|        results|violations|                                                     _errors|
+-------------+-----------------------+-----+---------------+----------+------------------------------------------------------------+
|      2633528|             FRESH BREW|60659|Out of Business|      NULL|[{has_violations, Column 'violations' value is null or em...|
|      2633530|TRADER JOE'S STORE #860|60659|      Not Ready|      NULL|[{has_violations, Column 'violations' value is null or em...|
|      2633488|      EASY STREET PIZZA|60634|           Pass|      NULL|[{has_violations, Column 'violations' value is null or em...|
|      2633458|     

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule
from databricks.labs.dqx.check_funcs import is_not_null_and_not_empty, regex_match, sql_expression
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

ws = WorkspaceClient()
dq_engine = DQEngine(ws)

dallas_bronze = spark.table("food_inspections.bronze.dallas_inspections")

violation_count_expr = sum(
    F.when(
        (F.col(f"violation_description_{i}").isNotNull()) &
        (F.trim(F.col(f"violation_description_{i}")) != ""),
        1
    ).otherwise(0)
    for i in range(1, 26)
)
dallas_bronze = dallas_bronze.withColumn("raw_violation_count", violation_count_expr)

dallas_checks = [
    DQRowRule(name="restaurant_name_not_null", criticality="error", check_func=is_not_null_and_not_empty, column="restaurant_name"),
    DQRowRule(name="inspection_date_not_null", criticality="error", check_func=is_not_null_and_not_empty, column="inspection_date"),
    DQRowRule(name="inspection_type_not_null", criticality="error", check_func=is_not_null_and_not_empty, column="inspection_type"),
    DQRowRule(name="zip_valid_format", criticality="error", check_func=regex_match, column="zip_code", check_func_kwargs={"regex": "^[0-9]{5}$"}),
    DQRowRule(name="score_valid_range", criticality="error", check_func=sql_expression, check_func_kwargs={"expression": "inspection_score IS NOT NULL AND CAST(inspection_score AS INT) >= 0 AND CAST(inspection_score AS INT) <= 100"}),
    DQRowRule(name="has_violations", criticality="error", check_func=sql_expression, check_func_kwargs={"expression": "raw_violation_count > 0"}),
    DQRowRule(name="score90_max3_violations", criticality="error", check_func=sql_expression, check_func_kwargs={"expression": "NOT (CAST(inspection_score AS INT) >= 90 AND raw_violation_count > 3)"}),
]

dallas_valid, dallas_invalid = dq_engine.apply_checks_and_split(dallas_bronze, dallas_checks)

valid_count = dallas_valid.count()
invalid_count = dallas_invalid.count()
print(f"Dallas valid:   {valid_count:,}")
print(f"Dallas invalid: {invalid_count:,}")
print(f"Total:          {valid_count + invalid_count:,}")

dallas_invalid.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("food_inspections.silver.dallas_quarantine")
print(f"\nQuarantine saved: food_inspections.silver.dallas_quarantine ({invalid_count:,} rows)")

print("\nError distribution:")
dallas_invalid.select(F.explode("_errors").alias("error")).select("error.name").groupBy("name").count().orderBy(F.desc("count")).show(truncate=False)

Dallas valid:   53,922
Dallas invalid: 25,062
Total:          78,984

Quarantine saved: food_inspections.silver.dallas_quarantine (25,062 rows)

Error distribution:
+------------------------+-----+
|name                    |count|
+------------------------+-----+
|score90_max3_violations |18301|
|has_violations          |6579 |
|zip_valid_format        |259  |
|restaurant_name_not_null|11   |
|score_valid_range       |6    |
+------------------------+-----+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Inspection type mapping
inspection_type_mapping = {
    "CANVAS": "CANVASS",
    "CANVASS REINSPECTION": "CANVASS RE-INSPECTION",
    "LICENSE REINSPECTION": "LICENSE RE-INSPECTION",
    "LICENSE RE-INSPECTON": "LICENSE RE-INSPECTION",
    "COMPLAINT REINSPECTION": "COMPLAINT RE-INSPECTION",
    "COMPLIANT": "COMPLAINT",
    "COMPLIANT RE-INSPECTION": "COMPLAINT RE-INSPECTION",
    "OUT OFBUSINESS": "OUT OF BUSINESS",
    "OUT OF BUSINES": "OUT OF BUSINESS",
    "OUTOFBUSINESS": "OUT OF BUSINESS",
    "NONINSPECTION": "NON-INSPECTION",
    "SHORTFORM COMPLAINT": "SHORT FORM COMPLAINT",
    "NOENTRY": "NO ENTRY",
}

type_expr = F.upper(F.trim(F.col("inspection_type")))
for bad, good in inspection_type_mapping.items():
    type_expr = F.when(F.upper(F.trim(F.col("inspection_type"))) == bad, F.lit(good)).otherwise(type_expr)

score_expr = (
    F.when(F.upper(F.trim(F.col("results"))) == "PASS", 90)
     .when(F.upper(F.trim(F.col("results"))) == "PASS W/ CONDITIONS", 80)
     .when(F.upper(F.trim(F.col("results"))) == "FAIL", 70)
     .when(F.upper(F.trim(F.col("results"))) == "NO ENTRY", 0)
     .otherwise(F.lit(None).cast("double"))
)

pass_fail_expr = (
    F.when(F.upper(F.trim(F.col("results"))).isin("PASS", "PASS W/ CONDITIONS"), "PASS")
     .when(F.upper(F.trim(F.col("results"))) == "FAIL", "FAIL")
     .otherwise("OTHER")
)

chicago_cleaned = chicago_valid.select(
    F.trim(F.col("inspection_id")).alias("inspection_id"),
    F.upper(F.trim(F.col("dba_name"))).alias("dba_name"),
    F.upper(F.trim(F.col("aka_name"))).alias("aka_name"),
    F.trim(F.col("license_num")).alias("license_num"),
    F.upper(F.trim(F.col("facility_type"))).alias("facility_type"),
    F.trim(F.col("risk")).alias("risk"),
    F.upper(F.trim(F.col("address"))).alias("address"),
    F.upper(F.trim(F.col("city"))).alias("city"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.trim(F.col("zip")).alias("zip"),
    F.to_date(F.col("inspection_date"), "MM/dd/yyyy").alias("inspection_date"),
    type_expr.alias("inspection_type"),
    F.upper(F.trim(F.col("results"))).alias("results"),
    F.col("violations"),
    F.col("latitude").cast("double").alias("latitude"),
    F.col("longitude").cast("double").alias("longitude"),
    score_expr.alias("violation_score"),
    pass_fail_expr.alias("pass_fail_flag"),
    F.trim(F.col("license_num")).alias("restaurant_nk"),
    F.trim(F.col("inspection_id")).alias("inspection_id_nk"),
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

# Parse violations
chicago_exploded = chicago_cleaned.withColumn(
    "violation_raw", F.explode(F.split(F.col("violations"), "\\s*\\|\\s*"))
).filter(
    (F.col("violation_raw").isNotNull()) & (F.trim(F.col("violation_raw")) != "")
)

chicago_violations = chicago_exploded.select(
    F.col("inspection_id_nk"),
    F.col("violation_raw"),
    F.regexp_extract(F.col("violation_raw"), r"^(\d+)\.", 1).alias("violation_code"),
    F.regexp_extract(F.col("violation_raw"), r"^\d+\.\s*(.*?)(?:\s*-\s*Comments\s*:.*$|$)", 1).alias("violation_description"),
    F.regexp_extract(F.col("violation_raw"), r"-\s*Comments\s*:\s*(.*)", 1).alias("violation_text"),
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

w = Window.partitionBy("inspection_id_nk").orderBy("violation_code")
chicago_violations = chicago_violations.withColumn("violation_order", F.row_number().over(w))

# Critical flag
chicago_violations = chicago_violations.withColumn(
    "is_critical_flag",
    F.when(
        F.upper(F.col("violation_raw")).rlike("PRIORITY VIOLATION|PRIORITY FOUNDATION VIOLATION|CITATION ISSUED|CRITICAL"),
        F.lit("Y")
    ).otherwise(F.lit("N"))
)

# Rule 7: Drop pure PASS with critical violations
critical_per_inspection = chicago_violations.groupBy("inspection_id_nk").agg(
    F.sum(F.when(F.col("is_critical_flag") == "Y", 1).otherwise(0)).alias("critical_count")
)

chicago_with_critical = chicago_cleaned.join(
    critical_per_inspection,
    on="inspection_id_nk",
    how="inner"
)

rule7_count = chicago_with_critical.filter(
    (F.col("results") == "PASS") & (F.col("critical_count") > 0)
).count()

chicago_inspections = chicago_with_critical.filter(
    ~((F.col("results") == "PASS") & (F.col("critical_count") > 0))
).drop("critical_count")

# Add total_violations
violation_counts = chicago_violations.groupBy("inspection_id_nk").agg(F.count("*").alias("total_violations"))
chicago_inspections = chicago_inspections.join(violation_counts, on="inspection_id_nk", how="left")

# Filter violations to surviving inspections
surviving_ids = chicago_inspections.select("inspection_id_nk").distinct()
chicago_violations = chicago_violations.join(surviving_ids, on="inspection_id_nk", how="inner")

print(f"Chicago inspections after all rules: {chicago_inspections.count():,}")
print(f"Chicago violations: {chicago_violations.count():,}")
print(f"Rule 7 dropped: {rule7_count}")

Chicago inspections after all rules: 221,458
Chicago violations: 1,000,262
Rule 7 dropped: 427


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql import DataFrame

dallas_cleaned = dallas_valid.select(
    F.upper(F.trim(F.col("restaurant_name"))).alias("restaurant_name"),
    F.col("street_number"),
    F.col("street_name"),
    F.col("street_direction"),
    F.col("street_type"),
    F.col("street_unit"),
    F.upper(F.trim(F.col("street_address"))).alias("street_address"),
    F.trim(F.col("zip_code")).alias("zip_code"),
    F.upper(F.trim(F.col("inspection_type"))).alias("inspection_type"),
    F.to_date(F.col("inspection_date"), "MM/dd/yyyy").alias("inspection_date"),
    F.col("inspection_score").cast("int").alias("inspection_score"),
    F.when(F.col("inspection_score").cast("int") >= 70, "PASS").otherwise("FAIL").alias("results"),
    F.when(F.col("inspection_score").cast("int") >= 70, "PASS").otherwise("FAIL").alias("pass_fail_flag"),
    F.col("inspection_score").cast("double").alias("violation_score"),
    F.regexp_extract(F.col("lat_long_location"), r"\(\s*(-?[\d.]+)\s*,", 1).cast("double").alias("latitude"),
    F.regexp_extract(F.col("lat_long_location"), r",\s*(-?[\d.]+)\s*\)", 1).cast("double").alias("longitude"),
    *[F.col(f"violation_description_{i}") for i in range(1, 26)],
    *[F.col(f"violation_points_{i}") for i in range(1, 26)],
    *[F.col(f"violation_detail_{i}") for i in range(1, 26)],
    *[F.col(f"violation_memo_{i}") for i in range(1, 26)],
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

dallas_cleaned = dallas_cleaned.withColumn(
    "restaurant_nk",
    F.md5(F.concat(F.col("restaurant_name"), F.lit("||"), F.col("street_address"), F.lit("||"), F.col("zip_code")))
)
dallas_cleaned = dallas_cleaned.withColumn(
    "inspection_id_nk",
    F.md5(F.concat(F.col("restaurant_nk"), F.lit("||"), F.col("inspection_date").cast("string"), F.lit("||"), F.col("inspection_score").cast("string")))
)

# Unpivot violations
violation_dfs = []
for i in range(1, 26):
    viol_df = dallas_cleaned.select(
        F.col("inspection_id_nk"),
        F.lit(i).alias("violation_order"),
        F.col(f"violation_description_{i}").alias("violation_description_raw"),
        F.col(f"violation_points_{i}").alias("violation_points_raw"),
        F.col(f"violation_detail_{i}").alias("violation_detail"),
        F.col(f"violation_memo_{i}").alias("violation_memo"),
        F.col("source_city"),
        F.current_timestamp().alias("load_ts"),
    ).filter(
        (F.col(f"violation_description_{i}").isNotNull()) & (F.trim(F.col(f"violation_description_{i}")) != "")
    )
    violation_dfs.append(viol_df)

dallas_violations_raw = reduce(DataFrame.unionAll, violation_dfs)

dallas_violations = dallas_violations_raw.select(
    F.col("inspection_id_nk"),
    F.col("violation_order"),
    F.when(
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*(\d+)", 1) != "",
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*(\d+)", 1)
    ).otherwise(F.lit("UNK")).alias("violation_code"),
    F.when(
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*\d+\s+(.*)", 1) != "",
        F.regexp_replace(F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*\d+\s+(.*)", 1), "ø", "\u00b0")
    ).otherwise(
        F.regexp_replace(F.trim(F.col("violation_description_raw")), "ø", "\u00b0")
    ).alias("violation_description"),
    F.regexp_replace(F.col("violation_detail"), "ø", "\u00b0").alias("violation_detail"),
    F.regexp_replace(F.col("violation_memo"), "ø", "\u00b0").alias("violation_memo"),
    F.col("violation_points_raw").cast("int").alias("points_assigned"),
    F.when(F.col("violation_points_raw").cast("int") == 3, "Y").otherwise("N").alias("is_critical_flag"),
    F.col("source_city"),
    F.col("load_ts"),
)

# Rule 7 for Dallas: score >= 90 with critical violations
critical_per_inspection = dallas_violations.groupBy("inspection_id_nk").agg(
    F.sum(F.when(F.col("is_critical_flag") == "Y", 1).otherwise(0)).alias("critical_count"),
    F.count("*").alias("total_violations"),
)

dallas_with_stats = dallas_cleaned.join(critical_per_inspection, on="inspection_id_nk", how="inner")

rule7_count = dallas_with_stats.filter(
    (F.col("inspection_score") >= 90) & (F.col("critical_count") > 0)
).count()

dallas_inspections = dallas_with_stats.filter(
    ~((F.col("inspection_score") >= 90) & (F.col("critical_count") > 0))
)

# Filter violations to surviving inspections
surviving_ids = dallas_inspections.select("inspection_id_nk").distinct()
dallas_violations = dallas_violations.join(surviving_ids, on="inspection_id_nk", how="inner")

print(f"Dallas inspections after all rules: {dallas_inspections.count():,}")
print(f"Dallas violations: {dallas_violations.count():,}")
print(f"Rule 7 dropped: {rule7_count}")

Dallas inspections after all rules: 42,176
Dallas violations: 281,411
Rule 7 dropped: 11746


In [0]:
from pyspark.sql import functions as F

# === UNIFIED INSPECTIONS ===
chicago_insp_unified = chicago_inspections.select(
    F.col("inspection_id_nk"),
    F.col("restaurant_nk"),
    F.col("dba_name"),
    F.col("aka_name"),
    F.col("license_num").alias("license_number"),
    F.col("facility_type"),
    F.col("risk").alias("risk_level"),
    F.col("address"),
    F.col("city"),
    F.col("state"),
    F.col("zip"),
    F.col("inspection_date"),
    F.col("inspection_type"),
    F.col("results"),
    F.col("violation_score"),
    F.col("pass_fail_flag"),
    F.col("total_violations"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("source_city"),
    F.col("load_ts"),
)

dallas_insp_unified = dallas_inspections.select(
    F.col("inspection_id_nk"),
    F.col("restaurant_nk"),
    F.col("restaurant_name").alias("dba_name"),
    F.lit(None).cast("string").alias("aka_name"),
    F.lit(None).cast("string").alias("license_number"),
    F.lit(None).cast("string").alias("facility_type"),
    F.lit(None).cast("string").alias("risk_level"),
    F.col("street_address").alias("address"),
    F.lit("DALLAS").alias("city"),
    F.lit("TX").alias("state"),
    F.col("zip_code").alias("zip"),
    F.col("inspection_date"),
    F.col("inspection_type"),
    F.col("results"),
    F.col("violation_score"),
    F.col("pass_fail_flag"),
    F.col("total_violations"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("source_city"),
    F.col("load_ts"),
)

unified_inspections = chicago_insp_unified.unionByName(dallas_insp_unified)

# === UNIFIED VIOLATIONS ===
chicago_viol_unified = chicago_violations.select(
    F.col("inspection_id_nk"),
    F.col("violation_order"),
    F.col("violation_code"),
    F.col("violation_description"),
    F.col("violation_text"),
    F.lit(None).cast("string").alias("violation_detail"),
    F.lit(None).cast("int").alias("points_assigned"),
    F.col("is_critical_flag"),
    F.col("source_city"),
    F.col("load_ts"),
)

dallas_viol_unified = dallas_violations.select(
    F.col("inspection_id_nk"),
    F.col("violation_order"),
    F.col("violation_code"),
    F.col("violation_description"),
    F.col("violation_memo").alias("violation_text"),
    F.col("violation_detail"),
    F.col("points_assigned"),
    F.col("is_critical_flag"),
    F.col("source_city"),
    F.col("load_ts"),
)

unified_violations = chicago_viol_unified.unionByName(dallas_viol_unified)

# Write unified tables
unified_inspections.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("food_inspections.silver.inspections")
unified_violations.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("food_inspections.silver.violations")

# Verify
ui = spark.table("food_inspections.silver.inspections")
uv = spark.table("food_inspections.silver.violations")

print(f"Unified inspections: {ui.count():,} rows, {len(ui.columns)} cols")
print(f"Unified violations:  {uv.count():,} rows, {len(uv.columns)} cols")

print(f"\nBy city:")
ui.groupBy("source_city").count().show()

# Integrity checks
zero_viol = ui.filter(F.col("total_violations") == 0).count()
orphan_viols = uv.join(ui.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
print(f"Inspections with 0 violations: {zero_viol}")
print(f"Orphan violations: {orphan_viols}")

print("\nUnified inspections schema:")
ui.printSchema()

Unified inspections: 263,634 rows, 21 cols
Unified violations:  1,281,673 rows, 10 cols

By city:
+-----------+------+
|source_city| count|
+-----------+------+
|    CHICAGO|221458|
|     DALLAS| 42176|
+-----------+------+

Inspections with 0 violations: 0
Orphan violations: 0

Unified inspections schema:
root
 |-- inspection_id_nk: string (nullable = true)
 |-- restaurant_nk: string (nullable = true)
 |-- dba_name: string (nullable = true)
 |-- aka_name: string (nullable = true)
 |-- license_number: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- risk_level: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- inspection_date: date (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- results: string (nullable = true)
 |-- violation_score: double (nullable = true)
 |-- pass_fail_flag: string (nullable = true)
 |-- total

In [0]:
from pyspark.sql import functions as F

ui = spark.table("food_inspections.silver.inspections")
uv = spark.table("food_inspections.silver.violations")
cq = spark.table("food_inspections.silver.chicago_quarantine")
dq = spark.table("food_inspections.silver.dallas_quarantine")

print("=" * 60)
print("SILVER LAYER SUMMARY (WITH DQX)")
print("=" * 60)

print(f"\n{'Table':<45} {'Rows':>10} {'Cols':>6}")
print("-" * 63)
print(f"{'silver.inspections (unified)':<45} {ui.count():>10,} {len(ui.columns):>6}")
print(f"{'silver.violations (unified)':<45} {uv.count():>10,} {len(uv.columns):>6}")
print(f"{'silver.chicago_quarantine':<45} {cq.count():>10,} {len(cq.columns):>6}")
print(f"{'silver.dallas_quarantine':<45} {dq.count():>10,} {len(dq.columns):>6}")

print("\n" + "=" * 60)
print("INSPECTIONS BY CITY")
print("=" * 60)
ui.groupBy("source_city").agg(
    F.count("*").alias("inspections"),
    F.sum("total_violations").alias("total_violations"),
    F.round(F.avg("violation_score"), 1).alias("avg_score"),
).show(truncate=False)

print("=" * 60)
print("DQX QUARANTINE BREAKDOWN")
print("=" * 60)

print("\nChicago quarantine reasons:")
cq.select(F.explode("_errors").alias("error")).select("error.name").groupBy("name").count().orderBy(F.desc("count")).show(truncate=False)

print("Dallas quarantine reasons:")
dq.select(F.explode("_errors").alias("error")).select("error.name").groupBy("name").count().orderBy(F.desc("count")).show(truncate=False)

print("=" * 60)
print("ROW REDUCTION FROM BRONZE")
print("=" * 60)
chi_bronze = 308357
dal_bronze = 78984
chi_silver = ui.filter(F.col("source_city") == "CHICAGO").count()
dal_silver = ui.filter(F.col("source_city") == "DALLAS").count()
print(f"\nChicago: {chi_bronze:,} -> {chi_silver:,} ({chi_bronze - chi_silver:,} dropped, {(chi_bronze - chi_silver)/chi_bronze*100:.1f}%)")
print(f"Dallas:  {dal_bronze:,} -> {dal_silver:,} ({dal_bronze - dal_silver:,} dropped, {(dal_bronze - dal_silver)/dal_bronze*100:.1f}%)")
print(f"Combined: {chi_bronze + dal_bronze:,} -> {ui.count():,}")

print("\n" + "=" * 60)
print("INTEGRITY CHECKS")
print("=" * 60)
zero_viol = ui.filter(F.col("total_violations") == 0).count()
orphan_viols = uv.join(ui.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
print(f"Inspections with 0 violations: {zero_viol}")
print(f"Orphan violations: {orphan_viols}")
print(f"All checks passed: {zero_viol == 0 and orphan_viols == 0}")

SILVER LAYER SUMMARY (WITH DQX)

Table                                               Rows   Cols
---------------------------------------------------------------
silver.inspections (unified)                     263,634     21
silver.violations (unified)                    1,281,673     10
silver.chicago_quarantine                         86,472     22
silver.dallas_quarantine                          25,062    120

INSPECTIONS BY CITY
+-----------+-----------+----------------+---------+
|source_city|inspections|total_violations|avg_score|
+-----------+-----------+----------------+---------+
|CHICAGO    |221458     |1000262         |82.6     |
|DALLAS     |42176      |281933          |87.9     |
+-----------+-----------+----------------+---------+

DQX QUARANTINE BREAKDOWN

Chicago quarantine reasons:
+------------------------+-----+
|name                    |count|
+------------------------+-----+
|has_violations          |86472|
|inspection_type_not_null|1    |
+-----------------------

In [0]:
%sql
SELECT source_city, COUNT(*) as violation_count
FROM food_inspections.silver.violations
GROUP BY source_city;

source_city,violation_count
CHICAGO,1000262
DALLAS,281411
